# BGE reranking: BM25 vs Dense vs Hybrid

Evaluates 152 source-article QA from `token_source_article_all`. Each strategy contributes its own fixed top-50 candidate pool; BGE only changes candidate order. Metrics are reported for all QA and `is_possible=True` QA separately.

In [ ]:
from pathlib import Path
import csv
import json
import time

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    raise RuntimeError('Run notebook from repository root.')

REPORT_PATH = REPO_ROOT / 'src/retrieval/output/token_source_article_all/per_query.csv'
CHUNKS_PATH = REPO_ROOT / 'src/chunking/output/vieonline_news_chunks_token.jsonl'
INPUT_DIR = REPO_ROOT / 'src/reranker/input/token_source_article_all'
OUTPUT_DIR = REPO_ROOT / 'src/reranker/output/token_source_article_all_bge'

MODEL = 'BAAI/bge-reranker-v2-m3'
CANDIDATE_K = 50
RERANK_TOP_K = 10  # Keep enough ranks for @1/@5/@10 comparison.
BATCH_SIZE = 8      # Lower to 4 if GPU runs out of memory.
MAX_LENGTH = 1024   # Lower to 512 only when memory is limited.
STRATEGIES = ('bm25', 'dense', 'hybrid')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Create one BGE input per retrieval strategy

Adds chunk text to each `retrieved_chunk_id`. Do not merge strategy pools: each BGE run must rerank its own top-50 list.

In [ ]:
from src.reranker.prepare_retrieval_report_inputs import build_inputs, write_inputs

inputs = build_inputs(REPORT_PATH, CHUNKS_PATH)
assert set(inputs) == set(STRATEGIES)
assert all(len(inputs[strategy]) == 152 for strategy in STRATEGIES)
assert all(
    len(row['candidates']) == CANDIDATE_K
    for strategy in STRATEGIES for row in inputs[strategy]
)
write_inputs(inputs, INPUT_DIR)
pd.DataFrame({
    'strategy': STRATEGIES,
    'queries': [len(inputs[s]) for s in STRATEGIES],
    'candidates_per_query': [len(inputs[s][0]['candidates']) for s in STRATEGIES],
})

## 2. Rerank each fixed top-50 pool with BGE

First run downloads `BAAI/bge-reranker-v2-m3`. Each output preserves gold source articles, answerability, original rank, and BGE score.

In [ ]:
from src.reranker.bge_reranker import rerank_rows

reranked_by_strategy = {}
rerank_latency_ms = {}
for strategy in STRATEGIES:
    started = time.perf_counter()
    reranked_rows = rerank_rows(
        inputs[strategy], MODEL, RERANK_TOP_K, BATCH_SIZE, MAX_LENGTH
    )
    rerank_latency_ms[strategy] = 1000 * (time.perf_counter() - started) / len(reranked_rows)
    reranked_by_strategy[strategy] = reranked_rows
    output_path = OUTPUT_DIR / f'{strategy}_bge_top{RERANK_TOP_K}.jsonl'
    with output_path.open('w', encoding='utf-8') as handle:
        for row in reranked_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'{strategy}: {len(reranked_rows)} queries, {rerank_latency_ms[strategy]:.1f} ms/query')

## 3. Evaluate article-level retrieval after reranking

Evaluation deduplicates result articles by first reranked chunk, matching retrieval report semantics. `all` measures source-document retrieval; `answerable` measures evidence retrieval for RAG.

In [ ]:
from src.retrieval.metrics import metrics_for_ranking

def unique_article_ids(candidates):
    seen, ranked = set(), []
    for candidate in candidates:
        article_id = str(candidate['article_id'])
        if article_id not in seen:
            seen.add(article_id)
            ranked.append(article_id)
    return ranked

def evaluate_rows(rows):
    scored = []
    for row in rows:
        metrics = metrics_for_ranking(
            set(map(str, row['gold_article_ids'])),
            unique_article_ids(row['reranked_candidates']),
        )
        scored.append({**metrics, 'is_possible': bool(row['is_possible'])})
    return pd.DataFrame(scored)

summary_rows = []
for strategy in STRATEGIES:
    detail = evaluate_rows(reranked_by_strategy[strategy])
    for split, subset in [('all', detail), ('answerable', detail[detail['is_possible']])]:
        summary_rows.append({
            'strategy': strategy,
            'split': split,
            'queries': len(subset),
            **subset.drop(columns='is_possible').mean().to_dict(),
            'mean_bge_rerank_latency_ms': rerank_latency_ms[strategy],
        })

summary = pd.DataFrame(summary_rows).sort_values(['split', 'ndcg@10'], ascending=[True, False])
summary.to_csv(OUTPUT_DIR / 'bge_rerank_summary.csv', index=False)
display(summary.round(4))

## 4. Compare before vs after BGE

Compare same split and metric. BGE can improve rank only when relevant source article already exists in its top-50 candidate pool.

In [ ]:
before = pd.read_csv(REPORT_PATH)
before = before[before['method'].isin(STRATEGIES)].copy()
before['split'] = before['is_possible'].map({True: 'answerable', False: 'unanswerable'})
before_all = before.assign(split='all')
before_summary = pd.concat([before, before_all]).groupby(['method', 'split'], as_index=False)[
    ['hit@1', 'hit@5', 'hit@10', 'recall@1', 'recall@5', 'recall@10', 'mrr@10', 'ndcg@10']
].mean().rename(columns={'method': 'strategy'})

after_summary = summary.drop(columns=['queries', 'mean_bge_rerank_latency_ms'])
comparison = before_summary.merge(after_summary, on=['strategy', 'split'], suffixes=('_before', '_after'))
for metric in ['hit@1', 'hit@5', 'hit@10', 'mrr@10', 'ndcg@10']:
    comparison[f'{metric}_delta'] = comparison[f'{metric}_after'] - comparison[f'{metric}_before']
comparison.to_csv(OUTPUT_DIR / 'bge_before_after_comparison.csv', index=False)
display(comparison.sort_values(['split', 'strategy']).round(4))